In [ ]:
!pip install kaggle==1.5.16

In [ ]:
! chmod 600 .kaggle/kaggle.json

In [ ]:
! kaggle competitions download Walmart-Recruiting-Store-Sales-Forecasting

In [ ]:
! unzip Walmart-Recruiting-Store-Sales-Forecasting.zip

In [ ]:
! unzip features.csv.zip
! unzip test.csv.zip
! unzip train.csv.zip

# Prophet — Walmart Store Sales Forecasting

**Model:** Prophet (Meta) — decomposable statistical model: trend + yearly seasonality + holiday effects.

**Key differences from the DL notebooks:**
- Prophet fits **one model per Store-Dept series** (no shared learning across series)
- Runs on **CPU** — no GPU needed
- Must be **fully refit for every rolling-origin fold** (cannot forecast from new context like DL models)

**Scope decision (logged in W&B):** hyperparameter search and evaluation run on the
**top 300 Store-Dept series by total sales**. Fitting all ~3,300 series across
3 folds × 4 grid configs would require 40,000+ individual Prophet fits (many hours on
Colab CPU). The top-300 subset covers a large share of total sales volume and gives a
meaningful WMAE comparison signal, consistent with the assignment note that classical
models matter mainly for theoretical comparison.

**Rolling evaluation:** 3 folds (instead of 7 used for DL models) — Prophet's refit-per-fold
cost makes folds the most expensive axis. The 3 folds still span different parts of the
validation timeline including the Labor Day holiday week.

In [ ]:
!pip install prophet mlflow dagshub -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle, os, json, logging
import mlflow
import mlflow.pyfunc

from prophet import Prophet

# Silence prophet/cmdstanpy per-fit logging (thousands of fits ahead)
logging.getLogger('prophet').setLevel(logging.ERROR)
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)

MLFLOW_EXPERIMENT = 'Prophet_Training'
MLFLOW_MODEL_NAME = 'prophet_walmart_best'

TRAIN_PATH    = 'train.csv'
FEATURES_PATH = 'features.csv'
STORES_PATH   = 'stores.csv'

H           = 4      # forecast horizon (weeks) — same as all other models
N_WINDOWS   = 3      # rolling folds (reduced: Prophet refits per fold)
TOP_SERIES  = 300    # scope: top Store-Dept series by total sales
VAL_START   = '2012-04-01'

# MLflow on Dagshub — shared team tracking server
import dagshub
dagshub.init(repo_owner='ikvas22', repo_name='Walmart-Recruiting---Store-Sales-Forecasting', mlflow=True)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

print('Setup OK')


## 1. EDA / Pre-processing

In [ ]:
with mlflow.start_run(run_name='Prophet_EDA'):
    mlflow.set_tag('stage', 'eda')

    train_raw    = pd.read_csv(TRAIN_PATH,    parse_dates=['Date'])
    features_raw = pd.read_csv(FEATURES_PATH, parse_dates=['Date'])
    stores_raw   = pd.read_csv(STORES_PATH)

    df = (
        train_raw
        .merge(features_raw, on=['Store', 'Date', 'IsHoliday'], how='left')
        .merge(stores_raw,   on=['Store'],                       how='left')
    )

    mlflow.log_metrics({
        'raw_rows' : float(df.shape[0]),
        'raw_cols' : float(df.shape[1]),
        'stores'   : float(df['Store'].nunique()),
        'depts'    : float(df['Dept'].nunique()),
    })
    mlflow.log_params({
        'date_min' : str(df['Date'].min().date()),
        'date_max' : str(df['Date'].max().date()),
    })

    # Prophet needs per-series frames with columns ds / y
    df['unique_id'] = df['Store'].astype(str) + '_' + df['Dept'].astype(str)
    df_nf = (
        df[['unique_id', 'Date', 'Weekly_Sales', 'IsHoliday']]
        .rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})
        .sort_values(['unique_id', 'ds'])
        .reset_index(drop=True)
    )

    # ── Scope: top TOP_SERIES series by total sales ────────────────
    series_totals = df_nf.groupby('unique_id')['y'].sum().sort_values(ascending=False)
    top_ids       = series_totals.head(TOP_SERIES).index
    df_nf         = df_nf[df_nf['unique_id'].isin(top_ids)].reset_index(drop=True)

    total_sales_share = series_totals.head(TOP_SERIES).sum() / series_totals.sum()

    mlflow.log_params({
        'scope_top_series' : TOP_SERIES,
        'val_start'        : VAL_START,
        'horizon_weeks'    : H,
        'n_windows'        : N_WINDOWS,
    })
    mlflow.log_metrics({
        'scope_total_sales_share' : float(total_sales_share),
        'scoped_rows'             : float(len(df_nf)),
    })

    print(f'Scoped to top {TOP_SERIES} series — covering {total_sales_share:.1%} of total sales volume')

    # ── Walmart holiday calendar for Prophet's native holiday handling ──
    holidays_df = pd.DataFrame([
        {'holiday': 'super_bowl',   'ds': '2010-02-12'},
        {'holiday': 'super_bowl',   'ds': '2011-02-11'},
        {'holiday': 'super_bowl',   'ds': '2012-02-10'},
        {'holiday': 'labor_day',    'ds': '2010-09-10'},
        {'holiday': 'labor_day',    'ds': '2011-09-09'},
        {'holiday': 'labor_day',    'ds': '2012-09-07'},
        {'holiday': 'thanksgiving', 'ds': '2010-11-26'},
        {'holiday': 'thanksgiving', 'ds': '2011-11-25'},
        {'holiday': 'christmas',    'ds': '2010-12-31'},
        {'holiday': 'christmas',    'ds': '2011-12-30'},
    ])
    holidays_df['ds'] = pd.to_datetime(holidays_df['ds'])
    holidays_df['lower_window'] = -1
    holidays_df['upper_window'] = 1

    df_train = df_nf[df_nf['ds'] <  VAL_START].copy()
    df_val   = df_nf[df_nf['ds'] >= VAL_START].copy()
    df_full  = pd.concat([df_train, df_val]).sort_values(['unique_id', 'ds']).reset_index(drop=True)

    mlflow.log_metrics({
        'train_rows'     : float(len(df_train)),
        'val_rows'       : float(len(df_val)),
        'holiday_events' : float(len(holidays_df)),
    })

    print(f'Train : {df_train["ds"].min().date()} → {df_train["ds"].max().date()}  ({len(df_train):,} rows)')
    print(f'Val   : {df_val["ds"].min().date()} → {df_val["ds"].max().date()}  ({len(df_val):,} rows)')


## 2. Training

**Note on skipped stages:** Feature_Engineering and Feature_Selection runs are
intentionally omitted for Prophet — the model consumes only the raw (ds, y)
series plus the holiday calendar. There are no engineered features to build
or select, so those MLflow runs would be empty.

Rolling-origin evaluation with the same window logic as the DL notebooks
(`N_WINDOWS` windows stepping back from the end in steps of `H`), except
Prophet is **refit from scratch on each fold's training data** — it cannot
forecast from a new context window without refitting.

Hyperparameter grid (4 configs) over the two most impactful Prophet knobs
for retail data:
- `changepoint_prior_scale` — trend flexibility (low = rigid, high = wiggly)
- `seasonality_mode` — additive vs multiplicative yearly seasonality

All configs use `yearly_seasonality=True`, `weekly_seasonality=False`
(data is already weekly), and the Walmart holiday calendar.


In [ ]:
def wmae(y_true: np.ndarray, y_pred: np.ndarray, is_holiday: np.ndarray) -> float:
    weights = np.where(is_holiday, 5.0, 1.0)
    return float(np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights))


def fold_cutoffs(df_source: pd.DataFrame, n_windows: int, h: int):
    """Fold cutoff dates mirroring the DL notebooks' rolling windows:
    walk back from the end of the timeline in steps of h."""
    dates = np.sort(df_source['ds'].unique())
    cutoffs = []
    for w in range(n_windows, 0, -1):
        cutoff_idx = len(dates) - w * h
        if cutoff_idx <= 0:
            continue
        cutoffs.append((dates[cutoff_idx - 1], dates[cutoff_idx:cutoff_idx + h]))
    return cutoffs


def evaluate_prophet_config(params: dict, df_source: pd.DataFrame,
                              n_windows: int = N_WINDOWS) -> tuple:
    """
    Rolling-origin evaluation for one Prophet config.
    For every fold and every series: refit Prophet on history up to the
    fold cutoff, forecast H weeks, score against actuals.
    """
    cutoffs   = fold_cutoffs(df_source, n_windows, H)
    all_preds = []

    for fold_i, (cutoff_date, target_dates) in enumerate(cutoffs):
        for uid in df_source['unique_id'].unique():
            s = df_source[df_source['unique_id'] == uid]
            hist   = s[s['ds'] <= cutoff_date][['ds', 'y']]
            target = s[s['ds'].isin(target_dates)]
            if len(hist) < 52 or target.empty:
                continue

            m = Prophet(
                yearly_seasonality      = True,
                weekly_seasonality      = False,
                daily_seasonality       = False,
                holidays                = holidays_df,
                changepoint_prior_scale = params['changepoint_prior_scale'],
                seasonality_mode        = params['seasonality_mode'],
            )
            m.fit(hist)

            future   = pd.DataFrame({'ds': target['ds'].values})
            forecast = m.predict(future)

            merged = target.merge(forecast[['ds', 'yhat']], on='ds', how='inner')
            for _, row in merged.iterrows():
                all_preds.append({
                    'fold'      : fold_i + 1,
                    'unique_id' : uid,
                    'ds'        : row['ds'],
                    'y'         : row['y'],
                    'y_pred'    : row['yhat'],
                    'IsHoliday' : row['IsHoliday'],
                })

    eval_df    = pd.DataFrame(all_preds)
    score_wmae = wmae(eval_df['y'].values, eval_df['y_pred'].values, eval_df['IsHoliday'].values)
    score_mae  = float(np.mean(np.abs(eval_df['y'].values - eval_df['y_pred'].values)))
    return score_wmae, score_mae, eval_df

In [ ]:
# ── Hyperparameter grid search — logged as the Training run ────
GRID = [
    {'changepoint_prior_scale': 0.05, 'seasonality_mode': 'additive'},
    {'changepoint_prior_scale': 0.05, 'seasonality_mode': 'multiplicative'},
    {'changepoint_prior_scale': 0.5,  'seasonality_mode': 'additive'},
    {'changepoint_prior_scale': 0.5,  'seasonality_mode': 'multiplicative'},
]

with mlflow.start_run(run_name='Prophet_Training') as training_run:
    mlflow.set_tag('stage', 'training')
    mlflow.log_params({
        'h'          : H,
        'n_windows'  : N_WINDOWS,
        'top_series' : TOP_SERIES,
        'grid'       : json.dumps(GRID),
    })

    best_wmae     = float('inf')
    best_mae      = None
    best_params   = None
    eval_best     = None
    trial_results = []

    for i, params in enumerate(GRID):
        print(f'\n── Config {i+1}/{len(GRID)}: {params}')
        trial_wmae, trial_mae, trial_eval = evaluate_prophet_config(params, df_full)
        print(f'   WMAE: {trial_wmae:,.2f}   MAE: {trial_mae:,.2f}')

        # Per-config metrics logged with a step index for comparison in the UI
        mlflow.log_metric(f'config_{i+1}_wmae', trial_wmae)
        mlflow.log_metric(f'config_{i+1}_mae',  trial_mae)
        trial_results.append({**params, 'wmae': trial_wmae, 'mae': trial_mae})

        if trial_wmae < best_wmae:
            best_wmae   = trial_wmae
            best_mae    = trial_mae
            best_params = params
            eval_best   = trial_eval

    trials_df = pd.DataFrame(trial_results)
    mlflow.log_text(trials_df.to_json(orient='records', indent=2), 'grid_results.json')

    mlflow.log_params({f'best_{k}': v for k, v in best_params.items()})
    mlflow.log_metrics({
        'best_val_wmae' : best_wmae,
        'best_val_mae'  : best_mae,
    })

    print(f'\n════════════════════════════════════')
    print(f'Best WMAE   : {best_wmae:,.2f}')
    print(f'Best params : {best_params}')

    # Prediction plots — logged as MLflow artifact
    sample_ids = eval_best['unique_id'].unique()[:6]
    fig, axes  = plt.subplots(3, 2, figsize=(14, 10))
    axes       = axes.flatten()

    for ax, uid in zip(axes, sample_ids):
        history = df_train[df_train['unique_id'] == uid].tail(52)
        actual  = eval_best[eval_best['unique_id'] == uid]
        ax.plot(history['ds'], history['y'],     label='History', color='steelblue')
        ax.plot(actual['ds'],  actual['y'],      label='Actual',  color='black')
        ax.plot(actual['ds'],  actual['y_pred'], label='Prophet', color='tomato', linestyle='--')
        hol = actual[actual['IsHoliday'] == 1]
        ax.scatter(hol['ds'], hol['y'], color='gold', zorder=5, s=40, label='Holiday')
        ax.set_title(f'Store-Dept {uid}', fontsize=10)
        ax.legend(fontsize=7)
        ax.tick_params(axis='x', rotation=30)

    plt.suptitle('Prophet Best Config — Rolling Validation Predictions', fontsize=13)
    plt.tight_layout()
    fig.savefig('prophet_predictions.png', dpi=100, bbox_inches='tight')
    mlflow.log_artifact('prophet_predictions.png')
    plt.show()

    # Per-series WMAE table — logged as MLflow artifact
    per_series = (
        eval_best
        .groupby('unique_id')
        .apply(lambda g: wmae(g['y'].values, g['y_pred'].values, g['IsHoliday'].values))
        .reset_index()
        .rename(columns={0: 'wmae'})
        .sort_values('wmae', ascending=False)
    )
    mlflow.log_text(per_series.to_json(orient='records', indent=2), 'per_series_wmae.json')
    print('Top 10 hardest series:')
    print(per_series.head(10).to_string(index=False))


## 3. Save Best Model to MLflow Registry

Prophet is per-series, so the registered "model" is a **pipeline wrapper**
that refits Prophet with the best hyperparameters on whatever raw series it
is given at predict time, then forecasts H weeks ahead. This satisfies the
assignment requirement that the registered pipeline runs directly on raw
unprocessed data.

In [ ]:
# MLflow connection already initialised in the setup cell (dagshub.init +
# set_experiment). Nothing to do here — kept as a visual section boundary.
print(f'Registering to experiment: {MLFLOW_EXPERIMENT}')


In [ ]:
# (see setup cell — mlflow.set_experiment already called)


In [ ]:
class ProphetWrapper(mlflow.pyfunc.PythonModel):

    def load_context(self, context):
        with open(context.artifacts['config'], 'r') as f:
            cfg = json.load(f)
        self.params      = cfg['params']
        self.h           = cfg['h']
        self.holidays_df = pd.DataFrame(cfg['holidays'])
        self.holidays_df['ds'] = pd.to_datetime(self.holidays_df['ds'])

    def predict(self, context, model_input: pd.DataFrame) -> pd.DataFrame:
        """
        Accepts raw DataFrame with [Store, Dept, Date, Weekly_Sales].
        Refits Prophet per series with the registered best hyperparameters
        and forecasts the next H weeks after each series' last observed date.
        """
        from prophet import Prophet
        import logging
        logging.getLogger('prophet').setLevel(logging.ERROR)
        logging.getLogger('cmdstanpy').setLevel(logging.ERROR)

        df_in = model_input.copy()
        df_in['Date']      = pd.to_datetime(df_in['Date'])
        df_in['unique_id'] = df_in['Store'].astype(str) + '_' + df_in['Dept'].astype(str)

        results = []
        for uid, group in df_in.groupby('unique_id'):
            g = group.sort_values('Date')
            hist = g[['Date', 'Weekly_Sales']].rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})
            if len(hist) < 52:
                continue   # need at least a year for yearly seasonality

            m = Prophet(
                yearly_seasonality      = True,
                weekly_seasonality      = False,
                daily_seasonality       = False,
                holidays                = self.holidays_df,
                changepoint_prior_scale = self.params['changepoint_prior_scale'],
                seasonality_mode        = self.params['seasonality_mode'],
            )
            m.fit(hist)

            last_date = g['Date'].max()
            future = pd.DataFrame({
                'ds': pd.date_range(start=last_date + pd.Timedelta(weeks=1),
                                     periods=self.h, freq='W-FRI')
            })
            forecast = m.predict(future)

            store, dept = uid.split('_')
            for _, row in forecast.iterrows():
                results.append({
                    'Store': int(store), 'Dept': int(dept),
                    'Date': row['ds'], 'Weekly_Sales_pred': float(row['yhat'])
                })

        return pd.DataFrame(results)


os.makedirs('mlflow_artifacts', exist_ok=True)
config_path = 'mlflow_artifacts/prophet_config.json'
with open(config_path, 'w') as f:
    json.dump({
        'params'   : best_params,
        'h'        : H,
        'holidays' : holidays_df.assign(ds=holidays_df['ds'].astype(str)).to_dict(orient='records'),
    }, f, indent=2)

with mlflow.start_run(run_name='Prophet_Best_Model') as run:
    mlflow.log_params({
        'changepoint_prior_scale' : best_params['changepoint_prior_scale'],
        'seasonality_mode'        : best_params['seasonality_mode'],
        'horizon'                 : H,
        'n_windows'               : N_WINDOWS,
        'scope_top_series'        : TOP_SERIES,
    })
    mlflow.log_metrics({
        'val_wmae' : best_wmae,
        'val_mae'  : best_mae,
    })
    mlflow.pyfunc.log_model(
        artifact_path         = 'prophet_model',
        python_model          = ProphetWrapper(),
        artifacts             = {'config': config_path},
        registered_model_name = MLFLOW_MODEL_NAME,
        pip_requirements      = ['prophet', 'pandas', 'numpy'],
    )
    print(f'Registered: {MLFLOW_MODEL_NAME}')
    print(f'Run ID    : {run.info.run_id}')
    print(f'Best WMAE : {best_wmae:,.2f}')

In [ ]:
loaded = mlflow.pyfunc.load_model(f'models:/{MLFLOW_MODEL_NAME}/latest')
# Full history for a couple of series (Prophet needs >= 52 weeks per series)
sample = train_raw[(train_raw['Store'] == 1) & (train_raw['Dept'].isin([1, 2]))][
    ['Store', 'Dept', 'Date', 'Weekly_Sales']
]
result = loaded.predict(sample)
print('Registry load OK')
print(result)